# Live Demo — Function Calling on NVIDIA NIM
*(Live class demo. Run every cell once before class; keep the executed copy open in a second tab as backup.)*

Three acts:
1. **The failure** — no tools, ask for live data
2. **The anatomy** — one tool, the full round trip, raw messages printed at every step
3. **The model decides + the cliffhanger** — no tool call when none is needed; single-turn helper can't chain

In [ ]:
%pip install -q openai
from getpass import getpass
from openai import OpenAI
import json

print("Enter your NVIDIA API key:")
client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=getpass(),
    timeout=90,
)
MODEL = "meta/llama-3.1-70b-instruct"   # 8B tool-calling is unreliable; 70B's latency = narration time
print("ready")

---
## Act 1 — What happens WITHOUT tools

The model's knowledge is frozen at its training cutoff (**parametric knowledge**). Watch:

In [ ]:
r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "What is HDFC Bank's current share price on the NSE?"}],
    max_tokens=150,
)
print(r.choices[0].message.content)

Either a refusal ("I don't have real-time data...") or a confidently stale number — both make the
point: **word association cannot fetch live data.** The fix is not a smarter model; it's a phone line
to your code.

---
## Act 2 — The anatomy of a tool call

One tool: `get_stock_price`. Note what backs it — **a five-entry Python dictionary.** The model
never executes anything; it can only *ask us* to run our own function. (In production this dict
would be a market-data API call. The wire format is identical.)

In [ ]:
# --- our "market data service" (demo prices, not real quotes) -----------------
PRICES = {
    "HDFCBANK": 1642.50,
    "RELIANCE": 2985.10,
    "INFY":     1856.75,
    "TCS":      4123.00,
    "SBIN":      834.20,
}

def get_stock_price(ticker):
    ticker = ticker.upper().strip()
    if ticker in PRICES:
        return {"ticker": ticker, "price_inr": PRICES[ticker]}
    return {"error": f"unknown ticker {ticker!r}", "known": list(PRICES)}

# --- how we DESCRIBE that function to the model (JSON Schema) ------------------
tools = [{
    "type": "function",
    "function": {
        "name": "get_stock_price",
        "description": "Get the latest share price in INR for a stock listed on the NSE.",
        "parameters": {
            "type": "object",
            "properties": {
                "ticker": {"type": "string",
                           "description": "NSE ticker symbol, e.g. HDFCBANK, RELIANCE"},
            },
            "required": ["ticker"],
        },
    },
}]
print("tool defined:", tools[0]["function"]["name"])

### Step 1 — the model responds with a *tool call*, not text

Remember Monday's messages format: dictionaries with `role` and `content`. Watch what comes back —
an assistant message whose **`content` is `None`**, carrying a `tool_calls` entry instead. This is
the fourth role's entrance.

In [ ]:
messages = [
    {"role": "system", "content": "You have a stock-price tool. Use it, and answer from its results."},
    {"role": "user", "content": "What is HDFC Bank's share price?"},
]

r1 = client.chat.completions.create(model=MODEL, messages=messages,
                                    tools=tools, tool_choice="auto")
msg = r1.choices[0].message
print("content   :", repr(msg.content))
tc = msg.tool_calls[0]
print("tool call :", tc.function.name)
print("arguments :", repr(tc.function.arguments), "  <-- a JSON *string*, not a dict!")

### Step 2 — WE run the function, and send the result back as a `tool` message

In [ ]:
args = json.loads(tc.function.arguments)        # string -> dict (a classic gotcha)
result = get_stock_price(**args)
print("our function returned:", result)

messages.append(msg)                             # the assistant's tool-call turn
messages.append({                                # the NEW role: "tool"
    "role": "tool",
    "tool_call_id": tc.id,
    "content": json.dumps(result),
})

print("\n--- the conversation so far ---")
for m in messages:
    role = m["role"] if isinstance(m, dict) else m.role
    content = m.get("content") if isinstance(m, dict) else m.content
    print(f"  {role:9s} | {str(content)[:80]}")

### Step 3 — call again; the model turns the data into an answer

In [ ]:
r2 = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
print(r2.choices[0].message.content)

That's the whole mechanism: **describe → the model requests → your code executes → the model
narrates.** Every agent you'll ever build is this loop with more steps.

> ⚠️ One NVIDIA-endpoint quirk worth knowing: it rejects **parallel** tool calls (asking for two
> prices in one turn can 400 with *"This model only supports single tool-calls at once"*). Real
> APIs have real constraints — production code guards for this.

In [ ]:
# OPTIONAL: provoke the parallel-tool-call constraint (fine if it errors -- that's the demo)
try:
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": "Answer using tools."},
                  {"role": "user", "content": "Get the prices of BOTH Reliance and Infosys."}],
        tools=tools, tool_choice="auto",
    )
    m = r.choices[0].message
    print("tool calls returned:", len(m.tool_calls or []))
except Exception as e:
    print("API rejected it:", e)

---
## Act 3 — The model *decides*, and the cliffhanger

`tool_choice="auto"` means the model chooses. Give it a question that needs no market data:

In [ ]:
r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "What is the capital of France?"}],
    tools=tools, tool_choice="auto",
)
m = r.choices[0].message
print("tool_calls:", m.tool_calls)
print("content   :", m.content)

No tool call — it just answers. Now the **cliffhanger**: a task where the second step depends on
the first step's *result*. This helper (it's the given code in Friday's exercise) allows ONE round
of tool calls, then must answer:

In [ ]:
def chat_with_tools_single_turn(user_message):
    """One round of tool calls, then one final (tool-free) response."""
    msgs = [{"role": "system", "content": "If you use tools, call at most ONE tool at a time."},
            {"role": "user", "content": user_message}]
    r = client.chat.completions.create(model=MODEL, messages=msgs,
                                       tools=tools, tool_choice="auto")
    m = r.choices[0].message
    if not m.tool_calls:
        return m.content
    msgs.append(m)
    for tc in m.tool_calls:
        result = get_stock_price(**json.loads(tc.function.arguments))
        print(f"[called get_stock_price -> {result}]")
        msgs.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps(result)})
    final = client.chat.completions.create(model=MODEL, messages=msgs)   # NOTE: no tools here
    return final.choices[0].message.content

print(chat_with_tools_single_turn(
    "Look up HDFC Bank's share price, then calculate how many whole shares "
    "I can buy with 500,000 rupees."))

It does the lookup, then has to do the arithmetic *itself* — by word association, the thing we
established on day one that it's bad at. Sometimes it's close; it's never *trustworthy*. What it
needs is a **calculator tool and the ability to loop**: act → observe → act again.

**That loop has a name — ReAct — and building it is Friday's exercise.**